In [3]:
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from jpeg_compression_detection import analyze_dct_spectrum, estimate_jpeg_quality
from jpeg_compression_detection.paths import DATA_DIR, GENERATED_DATA_DIR, display_path


# Convert one image to another file format


In [5]:
source_image_path = DATA_DIR / "test-images" / "00085_931230_pl.ppm"
single_conversion_dir = GENERATED_DATA_DIR / "single-blockiness-conversion"

if not source_image_path.is_file():
    print(f"Single-file conversion skipped; input is missing: {display_path(source_image_path)}")
else:
    single_conversion_dir.mkdir(parents=True, exist_ok=True)
    output_png_path = single_conversion_dir / f"{source_image_path.name}.png"
    with Image.open(source_image_path) as image:
        image.save(output_png_path)
    print(f"Saved: {display_path(output_png_path)}")


# Convert a series of camera-RAW files


In [ ]:
raw_camera_dir = DATA_DIR / "test-images"
raw_conversion_dir = GENERATED_DATA_DIR / "single-blockiness-raw-conversions"
raw_extensions = {".dng", ".nef", ".cr2", ".arw", ".raw", ".rw2", ".orf", ".pef"}
RUN_RAW_CONVERSION = False

if not RUN_RAW_CONVERSION:
    print("RAW conversion skipped. Set RUN_RAW_CONVERSION = True to enable it.")
elif not raw_camera_dir.is_dir():
    print(f"RAW conversion skipped; input directory is missing: {display_path(raw_camera_dir)}")
else:
    try:
        import imageio.v3 as iio
        import rawpy
    except ImportError as exc:
        print(f"RAW conversion dependency is unavailable: {exc}")
    else:
        raw_conversion_dir.mkdir(parents=True, exist_ok=True)
        for raw_path in sorted(raw_camera_dir.iterdir()):
            if not raw_path.is_file() or raw_path.suffix.lower() not in raw_extensions:
                continue
            try:
                with rawpy.imread(str(raw_path)) as raw_image:
                    rgb = raw_image.postprocess()
                for extension in (".png", ".jpeg", ".jpg", ".ppm"):
                    output_path = raw_conversion_dir / f"{raw_path.name}{extension}"
                    iio.imwrite(output_path, rgb)
                    print(f"Saved: {display_path(output_path)}")
            except Exception as exc:
                print(f"Unable to convert {display_path(raw_path)}: {exc}")


# JPEG and JPEG 2000 compression


In [7]:
compression_source_path = DATA_DIR / "test-images" / "00085_931230_pl.ppm"
format_conversion_dir = GENERATED_DATA_DIR / "single-blockiness-format-conversions"

if not compression_source_path.is_file():
    print(f"Compression example skipped; input is missing: {display_path(compression_source_path)}")
else:
    format_conversion_dir.mkdir(parents=True, exist_ok=True)
    with Image.open(compression_source_path) as image:
        rgb_image = image.convert("RGB")
        rgb_image.save(format_conversion_dir / f"{compression_source_path.name}.jpg", "JPEG", quality=85)
        rgb_image.save(format_conversion_dir / f"{compression_source_path.name}-100-percent.jpg", "JPEG", quality=100)
        try:
            image.save(format_conversion_dir / f"{compression_source_path.name}.jp2", "JPEG2000", quality_mode="rates", quality_layers=[0.3])
        except (OSError, ValueError) as exc:
            print(f"JPEG 2000 output skipped because Pillow lacks the required codec: {exc}")


# Explore blockiness with a Laplacian edge detector


In [ ]:
laplacian_image_path = DATA_DIR / "test-images" / "00085_931230_pl.ppm"
image = cv2.imread(str(laplacian_image_path)) if laplacian_image_path.is_file() else None

if image is None:
    print(f"Laplacian example skipped; image is missing or unreadable: {display_path(laplacian_image_path)}")
else:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title("Original image")
    plt.axis("off")

    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian_abs = np.uint8(np.absolute(laplacian))
    _, thresholded = cv2.threshold(laplacian_abs, 50, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresholded, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    marked_image = image.copy()
    cv2.drawContours(marked_image, contours, -1, (0, 0, 255), 1)

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(marked_image, cv2.COLOR_BGR2RGB))
    plt.title("Detected high-contrast contours")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# Define a single-boundary blockiness indicator


In [1]:
def detect_single_boundary_blockiness(image):
    """Return the blockiness ratio at the principal 7-to-8 pixel boundary."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) > 2 else image
    horizontal_difference = np.abs(gray[:, 1:] - gray[:, :-1])
    vertical_difference = np.abs(gray[1:, :] - gray[:-1, :])

    horizontal_block = horizontal_difference[:, 7::8]
    vertical_block = vertical_difference[7::8, :]
    horizontal_non_block = np.delete(horizontal_difference, np.arange(7, horizontal_difference.shape[1], 8), axis=1)
    vertical_non_block = np.delete(vertical_difference, np.arange(7, vertical_difference.shape[0], 8), axis=0)

    block_mean = (np.mean(horizontal_block) + np.mean(vertical_block)) / 2
    non_block_mean = (np.mean(horizontal_non_block) + np.mean(vertical_non_block)) / 2
    return block_mean / non_block_mean if non_block_mean > 0 else 0.0


def visualize_single_boundary_blockiness(image):
    """Return a heat map for the principal 8 x 8 block boundaries."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) > 2 else image
    blockiness_map = np.zeros_like(gray, dtype=float)
    for row in range(7, gray.shape[0], 8):
        if row + 1 < gray.shape[0]:
            blockiness_map[row, :] = np.abs(np.float32(gray[row, :]) - np.float32(gray[row + 1, :]))
    for column in range(7, gray.shape[1], 8):
        if column + 1 < gray.shape[1]:
            blockiness_map[:, column] += np.abs(np.float32(gray[:, column]) - np.float32(gray[:, column + 1]))
    return cv2.normalize(blockiness_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


def inspect_single_boundary_blockiness(image_path):
    """Display the scalar blockiness experiment for one existing image."""
    image_path = Path(image_path)
    image = cv2.imread(str(image_path)) if image_path.is_file() else None
    if image is None:
        print(f"Inspection skipped; image is missing or unreadable: {display_path(image_path)}")
        return None

    blockiness_ratio = detect_single_boundary_blockiness(image)
    energy_mean, energy_std = analyze_dct_spectrum(image)
    estimated_quality, psnr = estimate_jpeg_quality(image)
    is_jpeg = blockiness_ratio > 1.1

    plt.figure(figsize=(15, 10))
    plt.subplot(2, 2, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title("Original image")
    plt.axis("off")

    plt.subplot(2, 2, 2)
    plt.imshow(visualize_single_boundary_blockiness(image), cmap="hot")
    plt.title(f"Blockiness map (ratio: {blockiness_ratio:.2f})")
    plt.axis("off")

    grid_image = image.copy()
    height, width = image.shape[:2]
    for column in range(0, width, 8):
        cv2.line(grid_image, (column, 0), (column, height), (0, 255, 0), 1)
    for row in range(0, height, 8):
        cv2.line(grid_image, (0, row), (width, row), (0, 255, 0), 1)
    plt.subplot(2, 2, 3)
    plt.imshow(cv2.cvtColor(grid_image, cv2.COLOR_BGR2RGB))
    plt.title("8 x 8 block grid")
    plt.axis("off")

    result_text = "\n".join(
        [
            "Analysis results:",
            f"- JPEG blockiness threshold exceeded: {'YES' if is_jpeg else 'NO'}",
            f"- Boundary blockiness ratio: {blockiness_ratio:.2f}",
            f"- Mean DCT energy: {energy_mean:.2f}",
            f"- DCT-energy standard deviation: {energy_std:.2f}",
            f"- Estimated JPEG quality: {estimated_quality}%",
            f"- PSNR: {psnr:.2f} dB",
        ]
    )
    plt.subplot(2, 2, 4)
    plt.axis("off")
    plt.text(0.05, 0.5, result_text, fontsize=12)
    plt.tight_layout()
    plt.show()
    return is_jpeg, blockiness_ratio, estimated_quality, psnr


In [10]:
# Optional batch inspection.
# test_images_dir = DATA_DIR / "test-images"
# if test_images_dir.is_dir():
#     for image_path in sorted(test_images_dir.iterdir()):
#         inspect_single_boundary_blockiness(image_path)


In [ ]:
example_paths = [
    DATA_DIR / "test-images" / "small3.png",
    DATA_DIR / "thispersondoesnotexist-10k" / "0.jpg",
    DATA_DIR / "thispersondoesnotexist-10k" / "1.jfif",
    DATA_DIR / "jfif" / "person_0000.jfif",
    DATA_DIR / "jfif" / "person_0001.jfif",
    DATA_DIR / "test-images" / "00-swapped-woman.jpg",
]

available_examples = [path for path in example_paths if path.is_file()]
if not available_examples:
    print("Example inspection skipped because none of the optional images are available.")
else:
    for image_path in available_examples:
        inspect_single_boundary_blockiness(image_path)


# Optional data acquisition for indicator tests


In [12]:
DOWNLOAD_SYNTHETIC_FACES = False

if not DOWNLOAD_SYNTHETIC_FACES:
    print("Synthetic-face download skipped. Set DOWNLOAD_SYNTHETIC_FACES = True to enable it.")
else:
    import random
    import time

    import requests
    from tqdm.notebook import tqdm

    download_dir = GENERATED_DATA_DIR / "synthetic-face-downloads-single-blockiness"
    download_dir.mkdir(parents=True, exist_ok=True)
    for index in tqdm(range(10_000)):
        try:
            response = requests.get(
                "https://thispersondoesnotexist.com/",
                headers={"User-Agent": "research-reproduction/1.0"},
                timeout=30,
            )
            response.raise_for_status()
            (download_dir / f"person_{index + 1001:05d}.jfif").write_bytes(response.content)
            time.sleep(random.uniform(0.1, 0.3))
        except requests.RequestException as exc:
            print(f"Download failed for image {index}: {exc}")
    print("Download complete")


# Repeated JPEG compression experiments


## Convert baseline PPM images to PNG


In [13]:
RUN_PPM_TO_PNG_CONVERSION = False
multiple_compression_raw_dir = DATA_DIR / "multiple-compression"
multiple_compression_png_dir = GENERATED_DATA_DIR / "single-blockiness-png"

if not RUN_PPM_TO_PNG_CONVERSION:
    print("PPM-to-PNG conversion skipped. Set RUN_PPM_TO_PNG_CONVERSION = True to enable it.")
elif not multiple_compression_raw_dir.is_dir():
    print(f"PPM-to-PNG conversion skipped; source directory is missing: {display_path(multiple_compression_raw_dir)}")
else:
    multiple_compression_png_dir.mkdir(parents=True, exist_ok=True)
    for ppm_path in sorted(multiple_compression_raw_dir.glob("*.ppm")):
        try:
            with Image.open(ppm_path) as image:
                output_path = multiple_compression_png_dir / f"{ppm_path.stem}.png"
                image.save(output_path)
                print(f"Saved: {display_path(output_path)}")
        except Exception as exc:
            print(f"Unable to convert {display_path(ppm_path)}: {exc}")


## Three compression rounds with independently varied quality


In [ ]:
def generate_single_blockiness_dataset(raw_dir, work_dir, quality_levels, enabled=False):
    """Stage raw inputs and create up to three JPEG generations in a generated-data directory."""
    raw_dir = Path(raw_dir)
    work_dir = Path(work_dir)
    if not enabled:
        print(f"Generation skipped for {work_dir.name}; enable its RUN_* flag to execute it.")
        return
    if not raw_dir.is_dir():
        print(f"Generation skipped; raw input directory is missing: {display_path(raw_dir)}")
        return
    if work_dir.resolve().parent != GENERATED_DATA_DIR.resolve():
        raise RuntimeError(f"Refusing to modify a directory outside GENERATED_DATA_DIR: {display_path(work_dir)}")

    work_dir.mkdir(parents=True, exist_ok=True)
    for source_path in raw_dir.iterdir():
        if source_path.is_file() and source_path.suffix.lower() in {".png", ".ppm"}:
            shutil.copy2(source_path, work_dir / source_path.name)
    for path in work_dir.iterdir():
        if path.is_file() and path.suffix.lower() not in {".png", ".ppm"}:
            path.unlink()

    for png_path in sorted(work_dir.glob("*.png")):
        with Image.open(png_path) as image:
            rgb_image = image.convert("RGB")
            for quality in quality_levels:
                first_path = work_dir / f"{png_path.stem}_q{quality}.jpg"
                rgb_image.save(first_path, "JPEG", quality=quality)

    first_generation = sorted(work_dir.glob("*.jpg"))
    for first_path in first_generation:
        with Image.open(first_path) as first_image:
            rgb_first = first_image.convert("RGB")
            for second_quality in quality_levels:
                second_path = work_dir / f"{first_path.name}.q{second_quality}.jpg"
                rgb_first.save(second_path, "JPEG", quality=second_quality)
                with Image.open(second_path) as second_image:
                    for third_quality in quality_levels:
                        third_path = work_dir / f"{first_path.name}.q{second_quality}.q{third_quality}.jpg"
                        second_image.save(third_path, "JPEG", quality=third_quality)
    print(f"Generated repeated-compression data in {display_path(work_dir)}")


RUN_DIFFERENT_QUALITY_EXPERIMENT = False
different_quality_dir = GENERATED_DATA_DIR / "single-blockiness-different-quality"
generate_single_blockiness_dataset(
    DATA_DIR / "multiple-compression",
    different_quality_dir,
    [10, 30, 50, 70, 85, 95, 100],
    enabled=RUN_DIFFERENT_QUALITY_EXPERIMENT,
)


## Three compression rounds with the standard quality grid


In [ ]:
RUN_STANDARD_QUALITY_EXPERIMENT = False
standard_quality_dir = GENERATED_DATA_DIR / "single-blockiness-standard-quality"
generate_single_blockiness_dataset(
    DATA_DIR / "multiple-compression",
    standard_quality_dir,
    [10, 30, 50, 65, 75, 85, 95],
    enabled=RUN_STANDARD_QUALITY_EXPERIMENT,
)


## Calculate the scalar blockiness indicator


In [ ]:
RUN_SINGLE_BLOCKINESS_ANALYSIS = False
single_blockiness_results_path = GENERATED_DATA_DIR / "results_multiple_compression_single_blockiness.csv"
results = []

if not RUN_SINGLE_BLOCKINESS_ANALYSIS:
    print("Indicator analysis skipped. Set RUN_SINGLE_BLOCKINESS_ANALYSIS = True to enable it.")
elif not standard_quality_dir.is_dir():
    print(f"Indicator analysis skipped; input directory is missing: {display_path(standard_quality_dir)}")
else:
    image_paths = sorted(
        path for path in standard_quality_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".ppm"}
    )
    for image_path in image_paths:
        image = cv2.imread(str(image_path))
        if image is None:
            print(f"Unable to load image: {display_path(image_path)}")
            continue
        blockiness_ratio = detect_single_boundary_blockiness(image)
        energy_mean, energy_std = analyze_dct_spectrum(image)
        estimated_quality, psnr_value = estimate_jpeg_quality(image)
        results.append(
            {
                "filename": image_path.name,
                "blockiness_ratio": blockiness_ratio,
                "energy_mean": energy_mean,
                "energy_std": energy_std,
                "estimated_quality": estimated_quality,
                "psnr": psnr_value,
            }
        )

    if results:
        single_blockiness_results_path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(results).to_csv(single_blockiness_results_path, index=False)
        print(f"Saved {len(results)} rows to {display_path(single_blockiness_results_path)}")



Analyzing: multiple-compression\01 (1).png

Analyzing: multiple-compression\01 (1).ppm

Analyzing: multiple-compression\01 (1)_q10.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.q10.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.q30.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.q50.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.q65.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.q75.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.q85.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q10.q95.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q30.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q30.q10.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q30.q30.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q30.q50.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q30.q65.jpg

Analyzing: multiple-compression\01 (1)_q10.jpg.q30.q75.jpg

Analyzing: multiple-compression

## Present scalar-blockiness statistics


In [ ]:
results_df = (
    pd.read_csv(single_blockiness_results_path)
    if single_blockiness_results_path.is_file()
    else pd.DataFrame(results)
)

if results_df.empty:
    print(f"Summary skipped; result table is missing or empty: {display_path(single_blockiness_results_path)}")
else:
    def get_compression_count(filename):
        q_count = filename.count(".q")
        if filename.endswith((".png", ".ppm")):
            return 0
        return 1 if q_count == 0 else q_count + 1

    results_df["compression_count"] = results_df["filename"].apply(get_compression_count)
    compression_levels = sorted(results_df["compression_count"].unique())
    colors = ["blue", "green", "orange", "red"]
    figure, axes = plt.subplots(2, 2, figsize=(15, 10))

    blockiness_data = [results_df.loc[results_df["compression_count"] == level, "blockiness_ratio"].values for level in compression_levels]
    axes[0, 0].hist(blockiness_data, bins=30, label=[f"{level} compressions" for level in compression_levels], color=colors[: len(compression_levels)], edgecolor="black", stacked=True)
    axes[0, 0].set_xlabel("Principal-boundary blockiness ratio")
    axes[0, 0].set_ylabel("Number of images")
    axes[0, 0].set_title("Scalar blockiness distribution")
    axes[0, 0].axvline(1.1, color="black", linestyle="--", linewidth=2, label="JPEG threshold (1.1)")
    axes[0, 0].legend()

    groups = results_df.groupby("compression_count")["blockiness_ratio"].apply(list)
    axes[0, 1].boxplot(groups.values, labels=groups.index, patch_artist=True)
    axes[0, 1].set_xlabel("Number of compressions")
    axes[0, 1].set_ylabel("Principal-boundary blockiness ratio")
    axes[0, 1].set_title("Blockiness vs number of compressions")

    energy_data = [results_df.loc[results_df["compression_count"] == level, "energy_mean"].values for level in compression_levels]
    axes[1, 0].hist(energy_data, bins=30, label=[f"{level} compressions" for level in compression_levels], color=colors[: len(compression_levels)], edgecolor="black", stacked=True)
    axes[1, 0].set_xlabel("Mean DCT energy")
    axes[1, 0].set_ylabel("Number of images")
    axes[1, 0].set_title("DCT-energy distribution")
    axes[1, 0].legend()

    scatter = axes[1, 1].scatter(results_df["energy_mean"], results_df["energy_std"], c=results_df["compression_count"], cmap="plasma", alpha=0.6, s=20)
    axes[1, 1].set_xlabel("Mean DCT energy")
    axes[1, 1].set_ylabel("DCT-energy standard deviation")
    axes[1, 1].set_title("Mean vs standard deviation of DCT energy")
    plt.colorbar(scatter, ax=axes[1, 1], label="Number of compressions")
    plt.tight_layout()
    plt.show()

    print("\n=== Descriptive statistics ===")
    print(results_df[["blockiness_ratio", "estimated_quality", "psnr", "energy_mean", "energy_std"]].describe())
    print("\n=== Mean values by number of compressions ===")
    print(results_df.groupby("compression_count")[["blockiness_ratio", "estimated_quality", "psnr"]].mean())


## Analyze scalar blockiness across compression counts


In [ ]:
if results_df.empty:
    print("Compression-count analysis skipped because no results are available.")
else:
    grouped = results_df.groupby("compression_count")["blockiness_ratio"]
    summary = grouped.agg(["mean", "std", "count"])
    display(summary)

    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].errorbar(summary.index, summary["mean"], yerr=summary["std"], marker="o", capsize=4)
    axes[0].set_xlabel("Number of compressions")
    axes[0].set_ylabel("Mean principal-boundary blockiness ratio")
    axes[0].set_title("Mean blockiness by compression count")
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(
        [results_df.loc[results_df["compression_count"] == level, "blockiness_ratio"].values for level in compression_levels],
        bins=30,
        label=[f"{level} compressions" for level in compression_levels],
        stacked=True,
        edgecolor="black",
    )
    axes[1].set_xlabel("Principal-boundary blockiness ratio")
    axes[1].set_ylabel("Number of images")
    axes[1].set_title("Blockiness distribution by compression count")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
